# Methane from Space — Start Here
**Columbia University IEOR 4737 · ECB/EIB Sponsored Research · 2025–2026**

Run this notebook top-to-bottom before touching anything else. It executes the core computations live — calibration, validation, finance, and the risk API — and reads pre-computed JSON only for the timeseries, which requires the 211 GB Sentinel-2 tile cache to reproduce.

```
Sentinel-2 L1C imagery
  → CH4Net v8 inference        src/detection/ch4net_model.py
  → Conformal threshold τ      scripts/calibration/conformal_threshold.py
  → CEMF+IME quantification    src/quantification/
  → Annual timeseries          scripts/timeseries/
  → Monte Carlo Climate VaR    scripts/finance/finance_climate_var.py
  → FastAPI risk endpoints     src/api/main.py
```

## Step 1 — Environment
```bash
conda create -n methane python=3.11
conda activate methane
pip install -r requirements.txt
```

In [ ]:
import sys, json, math, re, subprocess, io, contextlib
from pathlib import Path

repo_root = Path.cwd()
while repo_root.name not in ('methane-api', '') and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / 'verify_outputs.py').exists():
    raise RuntimeError(f"Cannot find repo root. Launch Jupyter from ~/Downloads/methane-api/")

for p in ['', 'scripts/finance', 'scripts/validation',
          'scripts/calibration', 'scripts/quantification']:
    sys.path.insert(0, str(repo_root / p))

RESULTS = repo_root / 'results_analysis'
print(f"Repo root : {repo_root}")
print(f"Python    : {sys.version.split()[0]}")
print()
for pkg in ['numpy', 'torch', 'scipy', 'matplotlib', 'fastapi', 'uvicorn']:
    try:
        __import__(pkg); print(f"  ✓ {pkg}")
    except ImportError:
        print(f"  ✗ {pkg}  →  pip install -r requirements.txt")

## Step 2 — Output verification

`verify_outputs.py` runs 27 checks across every module: it opens each results JSON and checks that key numbers match the paper values within stated tolerances. If any check fails it exits non-zero and the cell below raises an error.

In [ ]:
result = subprocess.run(
    [sys.executable, str(repo_root / 'verify_outputs.py'), '--verbose'],
    capture_output=True, text=True, cwd=repo_root
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("verify_outputs.py failed — fix before proceeding.")

---
## Step 3 — Conformal Calibration  *(runs live)*

Loads the 35 raw S/C scores from non-emitter calibration sites and applies the split conformal prediction formula to derive τ. No pre-computed result is used here.

**Formula:** τ = the ⌈(n+1)(1−α)⌉-th order statistic of the non-emitter scores  
**Guarantee:** P(FPR ≤ α) for any exchangeable test set (Angelopoulos & Bates 2021)

Code: `scripts/calibration/conformal_threshold.py` · Full demo: `scripts/calibration/demo_calibration.ipynb`

In [ ]:
from conformal_threshold import ConformalCalibrator, NonEmitterScoreLoader

# Load the 35 raw S/C scores from non-emitter sites
loader = NonEmitterScoreLoader()
with contextlib.redirect_stdout(io.StringIO()):
    loader.Load()

scores = loader.scores
alpha  = 0.10
n      = len(scores)

# Run the calibrator
cal = ConformalCalibrator()
tau = cal.Fit(scores, alpha=alpha)
fpr = cal.EmpiricalFpr(scores, tau)

# Show the index calculation explicitly
idx = math.ceil((n + 1) * (1 - alpha))
print(f"n non-emitter sites  : {n}")
print(f"alpha                : {alpha}")
print(f"Order statistic index: ⌈({n}+1)(1−{alpha})⌉ = ⌈{(n+1)*(1-alpha):.1f}⌉ = {idx}")
print(f"Score at rank {idx:>2}     : {sorted(scores)[idx-1]:.4f}  ← τ")
print()
print(f"Empirical FPR at τ   : {fpr:.1%}  (≤ {alpha:.0%} guarantee satisfied)")
print(f"Sites exceeding τ    : {sum(1 for s in scores if s > tau)} of {n}")

assert abs(tau - 3.5796) < 0.001, f"tau mismatch: {tau}"
assert fpr <= alpha

---
## Step 4 — Annual Timeseries  *(reads pre-computed JSON)*

Running CH4Net v8 inference over all 147 Sentinel-2 acquisitions requires the 211 GB tile cache and ~hours of compute. The canonical output is `results_analysis/belchatow_annual_timeseries.json`. The cell below reads that file and plots the results.

To re-run from scratch: `caffeinate -i python scripts/timeseries/belchatow_annual_timeseries.py`

Code: `scripts/timeseries/belchatow_annual_timeseries.py` · Full demo: `scripts/timeseries/demo_timeseries.ipynb`

In [ ]:
with open(RESULTS / 'belchatow_annual_timeseries.json') as f:
    ts = json.load(f)

records    = ts['records']
quantified = [r for r in records if r.get('quantification', {}).get('status') == 'quantified']
flows      = [r['quantification']['flow_rate_kgh'] for r in quantified
              if r['quantification'].get('flow_rate_kgh') is not None]
mean_flow  = sum(flows) / len(flows)

print(f"Total acquisitions        : {len(records)}")
print(f"Quantification-supporting : {len(quantified)}  (paper: 30)")
print(f"Mean flow rate            : {mean_flow:.0f} kg/hr  (paper: 476 kg/hr)")
print(f"Flow range                : {min(flows):.0f}–{max(flows):.0f} kg/hr")
print(f"Annual estimate           : 4,174 t CH₄/yr  [95% CI: 2,987–5,360]")
print()
print("Note: June 2022 (sc_cfar=4.64) excluded per §6.3 — data quality issue.")
print("This is why above-threshold=31 but quantification-supporting=30.")

assert len(quantified) == 30
assert abs(mean_flow - 476) / 476 < 0.10

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

dates     = [datetime.fromisoformat(r['date']) for r in quantified]
flow_vals = [r['quantification']['flow_rate_kgh'] for r in quantified]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(dates, flow_vals, width=20, color='steelblue', alpha=0.8)
ax.axhline(mean_flow, color='red', ls='--', lw=1.5, label=f'Mean = {mean_flow:.0f} kg/hr')
ax.set_xlabel('Date')
ax.set_ylabel('Flow rate (kg/hr)')
ax.set_title('KWB Bełchatów — CH₄ emission rate per detection (2019–2024)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 5 — Validation  *(runs live)*

Three validators run directly from source. They read pre-computed score data (crop-level ML metrics and the timeseries JSON) but perform the statistical computations here — bootstrap resampling, leakage distance checks, leave-one-out stability.

Code: `scripts/validation/validation_metrics.py` · Full demo: `scripts/validation/demo_validation.ipynb`

In [ ]:
from validation_metrics import BootstrapAUROCValidator, LeakageAuditor, LooDetectionAnalyzer

# ── AUROC / Average Precision ──────────────────────────────────────────────
# Stratified bootstrap on 14 TROPOMI-confirmed real positive crops + 22 negatives.
# Synthetic crops are excluded. n_bootstrap=2000 matches the paper.
print("Running BootstrapAUROCValidator (n_bootstrap=2000) ...")
auroc_v = BootstrapAUROCValidator(n_bootstrap=2000)
with contextlib.redirect_stdout(io.StringIO()):
    auroc_r = auroc_v.Run()

a = auroc_r['auroc']
p = auroc_r['average_precision']
print(f"  AUROC : {a['point']:.3f}  90% CI [{a['ci_90_lo']:.3f}–{a['ci_90_hi']:.3f}]")
print(f"  AP    : {p['point']:.3f}  90% CI [{p['ci_90_lo']:.3f}–{p['ci_90_hi']:.3f}]")
print(f"  n_real_pos={auroc_r['n_real_positives']}  n_real_neg={auroc_r['n_real_negatives']}")
print()

# ── Leakage audit ──────────────────────────────────────────────────────────
# Checks temporal proximity between training crop dates and candidate site
# evaluation dates. Zero overlap required.
print("Running LeakageAuditor ...")
la = LeakageAuditor()
with contextlib.redirect_stdout(io.StringIO()):
    la_r = la.Run()
overlap = la_r.get('overlap_count', la_r.get('n_overlap', 0))
print(f"  Train/test date overlap : {overlap}")
print()

# ── LOO detection stability ────────────────────────────────────────────────
# Leave-one-out: remove each detection in turn and check detection rate is stable.
print("Running LooDetectionAnalyzer ...")
loo = LooDetectionAnalyzer()
with contextlib.redirect_stdout(io.StringIO()):
    loo_r = loo.Run()
print(f"  Stability verdict : {loo_r.get('stability_verdict', 'see results')}")
print()

assert a['point'] > 0.70
assert overlap == 0
print("✓ All validation checks pass")

---
## Step 6 — Monte Carlo Climate VaR  *(runs live)*

Runs the full 10,000-path Monte Carlo simulation from `ClimateVarEngine` and the deterministic CO₂e transition risk from `TransitionRiskAnalyzer`. Input parameters come from the paper (mean flow, CI, n_obs). Takes ~2 seconds.

Code: `scripts/finance/finance_climate_var.py` · `scripts/finance/finance_transition_risk.py` · Full demo: `scripts/finance/demo_finance.ipynb`

In [ ]:
import numpy as np
from finance_climate_var import (
    ClimateVarEngine, EmissionParams, UncertaintyParams, CarbonPriceParams
)
from finance_transition_risk import TransitionRiskAnalyzer, PGE_PROFILE, BELCHATOW_EVIDENCE

# ── Climate VaR — 10,000 Monte Carlo paths ─────────────────────────────────
# Inputs are the paper's emission parameters. Seed=42 for reproducibility.
ep = EmissionParams(
    mean_flow_kgh=476.0, sd_flow_kgh=363.0, n_obs=30,
    mean_annual_t=4174.0, ci95_lo=2987.0, ci95_hi=5360.0
)
engine = ClimateVarEngine(ep, UncertaintyParams(), CarbonPriceParams())

print("Running ClimateVarEngine (n_sim=10,000) ...")
with contextlib.redirect_stdout(io.StringIO()):
    results, em_draws, cp_draws, liab_draws = engine.Run(n_sim=10000, seed=42)

g100 = results['liability_gwp100']
g20  = results['liability_gwp20']

print(f"  {'Risk Metric':<38} {'GWP100 (M€)':>12}  {'GWP20 (M€)':>12}")
print("  " + "─" * 65)
print(f"  {'Mean expected liability':<38} {g100['mean']:>12.2f}  {g20['mean']:>12.2f}")
print(f"  {'99% Value-at-Risk':<38} {g100['var_99']:>12.2f}  {'—':>12}")
print(f"  {'99% Expected Shortfall':<38} {g100['es_99']:>12.2f}  {'—':>12}")

# ── Transition risk — deterministic CO₂e ──────────────────────────────────
print()
print("Running TransitionRiskAnalyzer ...")
analyzer = TransitionRiskAnalyzer(BELCHATOW_EVIDENCE, PGE_PROFILE)
with contextlib.redirect_stdout(io.StringIO()):
    tr = analyzer.Run()

print(f"  {'CO₂e GWP100 deterministic (t/yr)':<38} {tr['co2e_t_mean']:>12,.0f}  {'—':>12}")
print()
print(f"  Recovery vs Climate TRACE 2024: 14.1%  (reference: 29,636 t/yr)")

assert abs(g100['mean'] - 7.51) / 7.51 < 0.15
assert abs(tr['co2e_t_mean'] - 116872) / 116872 < 0.01

In [ ]:
# Plot the live liability distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(liab_draws, bins=100, color='steelblue', alpha=0.75, density=True)
ax.axvline(g100['mean'],   color='orange', ls='--', lw=2, label=f"Mean €{g100['mean']:.2f}M")
ax.axvline(g100['var_99'], color='red',    ls='--', lw=2, label=f"VaR99 €{g100['var_99']:.2f}M")
ax.set_xlabel('Annual carbon liability (M€, GWP100)')
ax.set_ylabel('Density')
ax.set_title('Monte Carlo Climate VaR — KWB Bełchatów (10,000 simulations)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 7 — CH4Net v8: model architecture  *(runs live)*

Instantiates the production U-Net architecture and confirms the parameter count. No weights file needed.

Code: `src/detection/ch4net_model.py` · Training: `scripts/detection/approach_c_retrain.py`

In [ ]:
from src.detection.ch4net_model import Unet

model       = Unet(in_channels=12, out_channels=1, div_factor=1)
total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Architecture : U-Net, div_factor=1")
print(f"Total params : {total_p:,}  (~13.5M — matches paper)")
print(f"Trainable    : {trainable_p:,}")
print()
print(f"{'Layer':<8} {'Type':<20} {'Params':>12}")
print("─" * 42)
for name, module in model.named_children():
    p = sum(x.numel() for x in module.parameters())
    print(f"{name:<8} {type(module).__name__:<20} {p:>12,}")

assert 13_000_000 < total_p < 14_000_000, f"Unexpected param count: {total_p:,}"

In [ ]:
# Check production weights status
prod_weights = repo_root / 'weights' / 'european_model_v8.pth'
if prod_weights.exists():
    mb = prod_weights.stat().st_size / (1 << 20)
    print(f"✓ european_model_v8.pth  ({mb:.0f} MB) — ready for inference")
else:
    print("✗ european_model_v8.pth not found")
    print("  Download: python download_weights.py")
    print("  (Set WEIGHTS_BASE_URL in download_weights.py first)")

### Re-training
Only needed if you extend the training dataset or add new sites.
Training data is already in `data/crops/` (41 MB, in git).

```bash
conda activate methane && cd ~/Downloads/methane-api

# Optional: generate more synthetic plumes
python scripts/quantification/generate_synthetic_plumes.py

# Optional: extract crops from new CDSE tiles
python scripts/quantification/extract_training_crops.py

# Fine-tune from Vaughan et al. base weights → outputs weights/european_model_v8.pth
python scripts/detection/approach_c_retrain.py --epochs 100 --lr 5e-5

# Dry run (1 epoch, no save)
python scripts/detection/approach_c_retrain.py --dry-run
```

Key config in `approach_c_retrain.py`: `FREEZE_DEPTH` (encoder blocks frozen), `POS_WEIGHT=15.0` (upweights plume pixels), `PATIENCE=20`.

---
## Step 8 — FastAPI  *(imports live, server launched separately)*

The API exposes the pipeline as REST endpoints for financial institution clients. The cell below confirms the module imports cleanly and lists all endpoints.

```bash
uvicorn src.api.main:app --reload --port 8000
# → http://localhost:8000/docs  (Swagger UI)
# → http://localhost:8000/health
```

Code: `src/api/main.py` · `src/api/schemas.py` · `src/api/risk_model.py`

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('schemas', repo_root / 'src/api/schemas.py')
schemas = importlib.util.module_from_spec(spec)
spec.loader.exec_module(schemas)
public = [x for x in dir(schemas) if not x.startswith('_') and x[0].isupper()]
print(f"schemas.py loaded — {len(public)} public classes")
print("  " + ", ".join(public[:6]) + (f"  ... +{len(public)-6} more" if len(public) > 6 else ""))
print()

api_src   = (repo_root / 'src/api/main.py').read_text()
endpoints = re.findall(r'@app\.(get|post|put|delete)\(["\']([^"\']+)["\']', api_src)
print(f"{'Method':<8} {'Endpoint'}")
print("─" * 45)
for method, path in endpoints:
    print(f"  {method.upper():<6} {path}")

assert len(endpoints) >= 5

---
## Step 9 — Full pipeline re-run  *(requires CDSE credentials + tile cache)*

Pre-computed outputs in `results_analysis/` are the canonical versions. Only re-run if you extend to new sites or time periods.

```bash
conda activate methane && cd ~/Downloads/methane-api

python download_weights.py                                         # ~52 MB
python scripts/detection/apply_bitemporal_diff.py --sites belchatow
python scripts/calibration/run_mac_inference.py --phase 2
python scripts/calibration/run_mac_inference.py --phase 3
caffeinate -i python scripts/timeseries/belchatow_annual_timeseries.py   # hours
python scripts/finance/finance_climate_var.py
python scripts/finance/finance_transition_risk.py
python scripts/validation/bootstrap_auroc_ap.py
python scripts/validation/leakage_audit.py
python scripts/validation/loo_detection_stability.py
python verify_outputs.py --verbose
```

---
## Step 10 — Demo notebooks

Each goes deeper into one module. All run offline from `results_analysis/`.

| Notebook | What it shows |
|---|---|
| `scripts/calibration/demo_calibration.ipynb` | Mondrian per-ecoregion extension, bootstrap CI on τ |
| `scripts/timeseries/demo_timeseries.ipynb` | Bełchatów + Rybnik class configs, site constants |
| `scripts/quantification/demo_quantification.ipynb` | 4-source uncertainty decomposition per detection |
| `scripts/validation/demo_validation.ipynb` | Full AUROC curves, held-out site evaluation |
| `scripts/finance/demo_finance.ipynb` | All 6 uncertainty layers, scenario sensitivity |

```bash
conda activate methane && cd ~/Downloads/methane-api
for nb in scripts/*/demo_*.ipynb; do
    jupyter nbconvert --to notebook --execute "$nb" --inplace
done
```

---
## Step 11 — Things to know before you extend this project

**1. The canonical timeseries is `results_analysis/belchatow_annual_timeseries.json`.**  
This is the merged 30-record dataset combining the 2021–2024 intensive monitoring run with 8 records from the 2019–2020 historical run. Both runs used the OSM mine polygon centroid (51.242°N, 19.275°E). Do not use the raw per-run JSONs in `results_analysis/timeseries/belchatow/` in isolation.

**2. June 2022 is excluded deliberately.**  
sc_cfar=4.64 (above τ) but excluded per §6.3 — data quality issue. This is why above-threshold=31 but quantification-supporting=30.

**3. div_factor=1, not div_factor=8.**  
`CH4NetDetector` auto-detects this from the weight file. Any code comment referencing div_factor=8 as "paper architecture" is from an early experimental phase — ignore it.

**4. Rybnik–Chwałowice not detecting is a finding, not a bug.**  
5 TROPOMI enhancements + 4 Carbon Mapper overpasses confirm emissions. Sub-threshold signals are present but the CFAR gate is inflated by Silesian terrain heterogeneity. Documented as a training distribution gap in §6.4.

**5. The 211 GB tile cache is not in git.**  
Re-download via CDSE. `src/ingestion/copernicus_client.py` handles this.

**6. `python verify_outputs.py --verbose` is your first diagnostic tool.**  
Run it after every change. 27 checks, all mapped to paper values.

**7. The API needs weights to serve `/detect` and `/site-risk`.**  
`/health` and `/events` work without weights.

---
*Columbia University · IEOR 4737 · Sponsored by ECB and EIB*